Using gpu

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# =========================
# 1. DEVICE (GPU)
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

# =========================
# 2. LOAD MODEL
# =========================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

# Load LoRA
model = PeftModel.from_pretrained(base_model, "v1-reply-model")

# Move to GPU
model = model.to(device)
model.eval()

print("✅ Model loaded successfully")

# =========================
# 3. BUILD PROMPT
# =========================

def build_prompt(conversation):
    return f"""### Instruction:
Generate exactly 3 short, natural reply suggestions.

### Rules:
- Only output 3 replies
- Each reply must be different
- Keep replies short (1 sentence)
- Do NOT add extra text

### Conversation:
{conversation}

### Response:
Reply 1:"""

# =========================
# 4. GENERATE REPLIES
# =========================

def generate_replies(conversation):
    prompt = build_prompt(conversation)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # =========================
    # 5. CLEAN OUTPUT
    # =========================

    # Extract only response part
    result = text.split("### Response:")[-1].strip()

    lines = [line.strip() for line in result.split("\n") if line.strip()]

    replies = []

    for line in lines:
        if line.startswith("Reply"):
            replies.append(line)

    # Stop at Reply 3
    clean_replies = []
    for r in replies:
        clean_replies.append(r)
        if "Reply 3:" in r:
            break

    # Ensure exactly 3 replies
    while len(clean_replies) < 3:
        clean_replies.append(f"Reply {len(clean_replies)+1}: ...")

    return clean_replies[:3]

# =========================
# 6. TEST
# =========================

if __name__ == "__main__":
    conversation = """User: I'm tired
Assistant: Long day?
User: Yeah"""

    replies = generate_replies(conversation)

    print("\n🔥 GENERATED REPLIES:\n")
    for r in replies:
        print(r)

🔥 Using device: cuda
✅ Model loaded successfully

🔥 GENERATED REPLIES:

Reply 1: "I know it can feel like a long day, but try to relax and take your time getting some rest."
Reply 2: "It doesn't have to last forever, you can get better sleep tonight. Try going to bed at the same time every night and see how that goes for you."
Reply 3: "Sometimes taking a nap or using a cooling shower or even just sitting in a quiet place with some calming music can help alle


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# =========================
# 1. DEVICE (GPU)
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

# =========================
# 2. LOAD MODEL
# =========================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = PeftModel.from_pretrained(base_model, "v1-reply-model")

model = model.to(device)
model.eval()

print("✅ Model loaded successfully")

# =========================
# 3. BUILD PROMPT
# =========================

def build_prompt(conversation):
    return f"""### Instruction:
Generate exactly 3 short replies.

### Rules:
- Each reply must be ONE short sentence
- No quotes
- No explanations
- No extra text
- Strict format:

Reply 1: ...
Reply 2: ...
Reply 3: ...

### Conversation:
{conversation}

### Response:
Reply 1:"""

# =========================
# 4. GENERATE REPLIES
# =========================

def generate_replies(conversation):
    prompt = build_prompt(conversation)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # =========================
    # 5. CLEAN OUTPUT
    # =========================

    result = text.split("### Response:")[-1].strip()

    lines = [line.strip() for line in result.split("\n") if line.strip()]

    replies = []

    for line in lines:
        if line.startswith("Reply"):
            # remove quotes
            line = line.replace('"', '')

            # keep only first sentence
            if "." in line:
                line = line.split(".")[0] + "."

            replies.append(line)

    # stop at Reply 3
    clean_replies = []
    for r in replies:
        clean_replies.append(r)
        if "Reply 3:" in r:
            break

    # ensure exactly 3 replies
    while len(clean_replies) < 3:
        clean_replies.append(f"Reply {len(clean_replies)+1}: ...")

    return clean_replies[:3]



In [ ]:
#  =========================
# 6. TEST
# =========================

if __name__ == "__main__":
#     conversation = """User: I'm tired
# Assistant: Long day?
# User: Yeah"""

    conversation = """User: Hey, what are you doing?
    Assistant: Just chilling
    User: Same here"""
    
    # conversation = "what is computer"

    # conversation = """User: I'm feeling really stressed
    # Assistant: What's going on?
    # User: Too much work lately"""


    # conversation = """User: You’re kinda funny
    # Assistant: Oh really?
    # User: Yeah, I like talking to you"""



    replies = generate_replies(conversation)

    print("\n🔥 GENERATED REPLIES:\n")
    
    #
for r in replies:
    print(r)


🔥 GENERATED REPLIES:

Reply 1: A computer is a device that uses digital electronics to perform arithmetic and logical operations on data.
Reply 2: ...
Reply 3: ...


🚀 BASE MODEL TEST CODE (NO FINETUNING)

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# =========================
# 1. DEVICE
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

# =========================
# 2. LOAD BASE MODEL ONLY
# =========================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

print("✅ Base model loaded")

# =========================
# 3. SIMPLE PROMPT
# =========================

def generate(conversation):
    prompt = f"""User: {conversation}
Assistant:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =========================
# 4. TEST INPUTS
# =========================

test_inputs = [
    "Hey, what are you doing?",
    "I'm feeling stressed",
    "You’re kinda funny"
]

# =========================
# 5. RUN TEST
# =========================

print("\n🔥 BASE MODEL OUTPUT:\n")

for text in test_inputs:
    print(f"👉 User: {text}")
    output = generate(text)
    print(f"{output}\n")

🔥 Using device: cuda
✅ Base model loaded

🔥 BASE MODEL OUTPUT:

👉 User: Hey, what are you doing?
User: Hey, what are you doing?
Assistant: I am working on improving my typing speed.
User: Ah, I see. How are you doing with that?
Assistant: I am improving gradually, but I have noticed a noticeable improvement in my speed.
User: I see. That's great to hear. Do you think you could help me with a task at work?
Assistant: I would be happy to help you with that. What task would you like me to do?
User: I

👉 User: I'm feeling stressed
User: I'm feeling stressed
Assistant: I understand that you may be feeling stressed, and here are some tips to help you relax:

1. Practice mindfulness: Take a few deep breaths and focus on the present moment. Try to observe your thoughts without judgment.

2. Exercise: Regular exercise can help reduce stress and improve overall health.

3. Get enough sleep: A good night's sleep can help you feel more rested and relaxed.

4.

👉 User: You’re kinda funny
User: You’